# 05 · Real microscope: step-by-step tests

Run this on the real microscope **from top to bottom, one section at a time**. Each level builds on the one before:

| Level | What | Moves the microscope? |
|---|---|---|
| 0 | Setup and safety switches | no |
| 1 | MCP, data and timing | no (takes images) |
| 2 | Individual nodes | only with `ALLOW_STAGE` |
| 3 | Instrument checks: metric response, corrector sign, beam x/y order | with `ALLOW_STAGE` / `ALLOW_CORRECTOR` / `ALLOW_BEAM` |
| 4 | Tuning nodes: focus z, C1, A1 | with `ALLOW_STAGE` / `ALLOW_CORRECTOR` |
| 5 | Segmentation and EDS | EDS with `ALLOW_BEAM` |
| 6 | Build your own graph | depends on the nodes you put in |
| 7 | Full run | all switches |

**Nothing moves unless you switch it on in the first cell.** A gated cell prints `SKIPPED` when its switch is off, so "Run All" is safe.

**Before you start**
- The asyncroscopy MCP server for the microscope is running, and `STEM_REAL_MCP_URL` is set in `.env`.
- Scan size, dwell time and EDS exposure are set by hand (not reachable through MCP yet).
- Dry run first: set `TARGET = "twin"` and start the twin (`uv run python scripts/twin.py start`). The whole notebook runs on the twin too.

## Level 0 · Setup and safety switches

In [ ]:
TARGET = "real"            # "real", or "twin" for a dry run

# Safety switches: nothing moves unless switched on here
ALLOW_STAGE = False        # stage moves: move node, z response, focus z
ALLOW_CORRECTOR = False    # corrector changes: C1 / A1 response, sign check, tune C1 / A1
ALLOW_BEAM = False         # park the beam and take EDS spectra

# Small, careful settings for a first session (defaults in stem_agent/config.py are larger)
TEST_TUNING = {
    "z":  {"step_m": 0.5e-6, "max_offset_m": 5e-6,  "max_iterations": 6},
    "C1": {"step_m": 5e-9,   "max_offset_m": 20e-9, "max_iterations": 6},
    "A1": {"step_m": 1e-9,   "max_offset_m": 4e-9,  "max_iterations": 6},
}
TUNING_FOV_M = 186e-9 if TARGET == "real" else 40e-9   # tuning, overview, segmentation
MAX_EDS_POINTS = 3

In [ ]:
%matplotlib inline
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from stem_agent.analysis.metrics import METRICS, all_metrics
from stem_agent.analysis.segmentation import segment
from stem_agent.config import TARGETS, TOOLS
from stem_agent.data import tiled_io
from stem_agent.mcp import session
from stem_agent.nodes import common
from stem_agent.nodes.eds_node import eds_node
from stem_agent.nodes.move_stage_node import move_stage_node, undo_move
from stem_agent.nodes.overview_node import overview_node
from stem_agent.nodes.preflight_node import preflight_node
from stem_agent.nodes.save_node import save_node
from stem_agent.nodes.segment_node import segment_node
from stem_agent.nodes.set_fov_node import set_fov_node, undo_fov
from stem_agent.nodes.status_node import (device_state, normalize_parameters, status_end_node,
                                          status_start_node)
from stem_agent.nodes.tune_node import (Corrector, StageZ, focus_z_node, tune_a1_node,
                                        tune_c1_node)
from stem_agent.utils.plots import plot_graph, plot_particles


def allowed(flag, what):
    """Gate for cells that change the microscope."""
    if not flag:
        print(f"SKIPPED: {what}. Switch it on in the first cell to run it.")
    return flag


def apply(state, update):
    """Merge a node's update into a new state dict (errors and tuning add up, as in the graph)."""
    print("->", sorted(update), "| run_status:", update.get("run_status", state.get("run_status")))
    for error in update.get("errors", []):
        print("ERROR:", error)
    merged = {**state, **update}
    if "errors" in update:
        merged["errors"] = state.get("errors", []) + update["errors"]
    if "tuning" in update:
        merged["tuning"] = {**state.get("tuning", {}), **update["tuning"]}
    return merged


def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5), squeeze=False)
    for ax, image, title in zip(axes[0], images, titles):
        ax.imshow(image, cmap="gray")
        ax.set_title(title)
        ax.axis("off")
    plt.show()


print(f"target {TARGET} | stage {ALLOW_STAGE} | corrector {ALLOW_CORRECTOR} | beam {ALLOW_BEAM}")

## Level 1 · MCP, data and timing (nothing moves)

### 1.1 Connect and check the tools
Every short name in `config.TOOLS` must exist on the server. Tool names are `<TangoClass>_<command>`.

In [ ]:
url = TARGETS[TARGET]["mcp_url"]
if not url:
    raise SystemExit("No MCP URL: set STEM_REAL_MCP_URL in .env and restart the kernel")
client = session.connect(TARGET)
tools = await client.list_tools()
print(f"{url}: {len(tools)} tools")
missing = []
for short in TOOLS:
    name = session.tool_name(short)
    tool = tools.get(name)
    args = ", ".join((tool.inputSchema or {}).get("properties", {})) if tool else ""
    print(f"  {'ok     ' if tool else 'MISSING'} {short:<20} {name}({args})")
    if tool is None:
        missing.append(short)
print("missing:", missing or "none")

### 1.2 If tools are missing: which Tango classes does the server have?
If the class names differ from `TARGETS[TARGET]`, fix them here (and later in `stem_agent/config.py`), then re-run 1.1.

In [ ]:
for command in ("acquire_scanned_image", "move_stage", "correct_aberration"):
    owners = [t.removesuffix("_" + command) for t in tools if t.endswith("_" + command)]
    print(f"{command:<24} offered by {owners}")
print("configured:", {k: TARGETS[TARGET][k] for k in ("microscope", "corrector")})
# e.g. TARGETS[TARGET]["corrector"] = "CEOSCorrector"

### 1.3 Read-only status
Check:
- **alpha**: if it reads `nan`, the 2.4 round-trip test matters.
- **`get_parameters` keys**: the status node reads `optical_mode`, `beam_blanker`, `column_valves`, `is_accelerator_on` and `beam_stopper`. A missing key means "unknown", and on the real microscope unknown stops the run.
- **device states**: each should be `ON`.

In [ ]:
stage = await common.get_stage()
fov = await session.call("get_fov")
defocus = await session.call("get_defocus")
print("stage [x, y, z, alpha, beta]:", stage, "| alpha is NaN:", np.isnan(stage[3]))
print(f"FOV {fov * 1e9:.1f} nm | defocus {defocus}")

try:
    params = await session.call("get_parameters")
    params = json.loads(params) if isinstance(params, str) else params
    print("get_parameters keys:", sorted(params))
    print("normalized for the status node:", normalize_parameters(params))
except Exception as exc:
    print("get_parameters FAILED:", exc)
    print("-> the status node cannot confirm readiness; fix this in asyncroscopy before a full run")

for tool in sorted(t for t in tools if t.endswith("_State")):
    print(f"  {tool.removesuffix('_State'):<24} {device_state(await session.call_tool(tool))}")

### 1.4 Tiled and one HAADF image
MCP returns the Tiled key; the full image comes from Tiled over HTTP.

In [ ]:
uri = json.loads(await session.call("get_tiled_config"))["uri"]
tiled_io.connect(uri)
t0 = time.time()
key, image = await common.acquire_haadf()
print(f"Tiled {uri} | key {key} | {image.shape} {image.dtype} | {time.time() - t0:.1f} s")
print({name: round(value, 4) for name, value in all_metrics(image).items()})
show([image], [f"HAADF, FOV {fov * 1e9:.0f} nm"])

### 1.5 Timing
MCP gives each command 30 s on the server side. Acquisitions must stay well below that.

In [ ]:
times = []
for _ in range(3):
    t0 = time.time()
    await common.acquire_haadf()
    times.append(time.time() - t0)
print(f"HAADF: {np.mean(times):.1f} s per image (min {min(times):.1f}, max {max(times):.1f}); limit 30 s")

## Level 2 · Individual nodes
Each node runs alone on a state built by hand, exactly as the graph will run it. `SETTINGS["target_xy_m"]` is the **current** position, so nothing moves by accident.

In [ ]:
x0, y0 = stage[0], stage[1]
SETTINGS = {
    "target": TARGET,
    "target_xy_m": (x0, y0),          # stay where we are
    "tuning_fov_m": TUNING_FOV_M,
    "tuning": TEST_TUNING,
    "max_particles": MAX_EDS_POINTS,
    "run_tag": "real_tests",
}
state = {"settings": SETTINGS}

### 2.1 Preflight
Expected: `run_status: running`, run folder created, settings checked.

In [ ]:
state = apply(state, await preflight_node(state))
state["run"]["run_dir"]

### 2.2 Status start
Expected: `ready: True`. **If not ready, stop here** and fix the microscope; the node never fixes anything itself.

In [ ]:
state = apply(state, await status_start_node(state))
snap = state["start"]
{k: snap[k] for k in ("stage", "fov_m", "stem_mode", "beam_blanked", "column_valves_open",
                      "high_tension_on", "beam_stopper_out", "devices", "ready", "problems", "warnings")}

### 2.3 Overview (no motion)

In [ ]:
state = apply(state, await overview_node(state))
Image(state["overview"]["preview_path"])

### 2.4 Stage round trip without moving
Sends the stage position it just read back to `move_stage`. If alpha is `NaN` and that can't pass through MCP's JSON, the error shows up here, **without any motion**.

In [ ]:
if allowed(ALLOW_STAGE, "stage round trip"):
    print("before:", await common.get_stage())
    print("after: ", await common.move_stage())     # no axis given: same position

### 2.5 Move node: +50 nm in x, then undo
Expected: readback within 10 nm, image shifted by 50 nm, back at the start.

In [ ]:
if allowed(ALLOW_STAGE, "move node"):
    trial = {**state, "settings": {**SETTINGS, "target_xy_m": (x0 + 50e-9, y0)}}
    _, before_image = await common.acquire_haadf()
    update = apply(trial, await move_stage_node(trial))
    _, after_image = await common.acquire_haadf()
    show([before_image, after_image], ["before", "after +50 nm in x"])
    await undo_move(trial)
    print("back at:", await common.get_stage())

### 2.6 Set FOV, then undo
Expected: readback within 1 %, wider view, back to the start FOV.

In [ ]:
before = state
state = apply(state, await set_fov_node(state))
_, zoom_image = await common.acquire_haadf()
await undo_fov(before)
state = apply(state, {"current": {**state["current"], "fov_m": await session.call("get_fov")}})
show([zoom_image], [f"FOV {TUNING_FOV_M * 1e9:.0f} nm"])
print(f"FOV back to {state['current']['fov_m'] * 1e9:.1f} nm")

## Level 3 · Instrument checks
These answer questions the twin can't:
- Does the focus metric respond on this sample?
- Which sign does the corrector use?
- Which beam x/y order is right?

Each cell restores what it changed.

### 3.1 Does the metric respond to stage z?
Five z values around the start, one image each. Expected: `contrast` and `edge_sharpness` peak near focus. **If the curves are flat, the search can't find focus**: use a larger step, or check the sample and FOV.

In [ ]:
def plot_response(offsets, rows, unit, unit_name, title):
    fig, ax = plt.subplots(figsize=(6, 4))
    for name in METRICS:
        values = np.array([row[name] for row in rows])
        ax.plot(np.array(offsets) / unit, values / values[len(values) // 2], "o-", label=name)
    ax.set(xlabel=f"offset ({unit_name})", ylabel="relative to the start point", title=title)
    ax.legend(frameon=False, fontsize=8)
    plt.show()


async def response(actuator, offsets):
    """Image at each offset; always back to the start afterwards."""
    await actuator.start()
    rows = []
    try:
        for offset in offsets:
            await actuator.apply(np.atleast_1d(offset).astype(float))
            rows.append(all_metrics((await common.acquire_haadf())[1]))
    finally:
        await actuator.restore()
    return rows


if allowed(ALLOW_STAGE, "z response"):
    step = TEST_TUNING["z"]["step_m"]
    offsets = [-2 * step, -step, 0.0, step, 2 * step]
    plot_response(offsets, await response(StageZ(), offsets), 1e-6, "um", "metrics vs stage z")

### 3.2 Does the metric respond to C1?
The same scan with the corrector.

In [ ]:
if allowed(ALLOW_CORRECTOR, "C1 response"):
    state = apply(state, await set_fov_node(state))   # C1 / A1 are tuned at the tuning FOV
    step = TEST_TUNING["C1"]["step_m"]
    offsets = [-2 * step, -step, 0.0, step, 2 * step]
    plot_response(offsets, await response(Corrector("C1"), offsets), 1e-9, "nm", "metrics vs C1")

### 3.3 Corrector sign
Uses the corrector's own measurement (`measure_c1a1`) before and after a +5 nm C1 step, then steps back. Compare the C1 values by eye: if C1 changed by **+5 nm** the sign is "adds"; if by **−5 nm** it is "subtracts" (the twin subtracts). The search doesn't need the sign, but you need it to read the results.

On the real CEOS a measurement can take a while; if it times out (30 s), skip this cell.

In [ ]:
if allowed(ALLOW_CORRECTOR, "corrector sign"):
    measure = f"{TARGETS[TARGET]['corrector']}_measure_c1a1"
    if measure not in tools:
        print("no", measure, "tool on this server")
    else:
        c1 = Corrector("C1")
        print("before:", await session.call_tool(measure))
        await c1.apply(np.array([5e-9]))
        print("after +5 nm:", await session.call_tool(measure))
        await c1.restore()
        print("after restore:", await session.call_tool(measure))

### 3.4 Beam x/y order
Takes a fresh overview, finds the largest particle and a background point, then takes EDS at both with each beam order. The right order shows **much more signal on the particle than on the background**:
- real microscope: total counts
- twin: the largest element fraction

The configured value is `TARGETS[TARGET]["beam_order"]`; the real one is still unverified.

In [ ]:
def signal(record):
    values = np.load(record["npy_path"])
    return float(values.max()) if record.get("elements") else float(values.sum())


if allowed(ALLOW_BEAM, "beam order check"):
    state = apply(state, await set_fov_node(state))
    probe = apply(state, await overview_node(state))
    overview = tiled_io.read_image(probe["overview"]["tiled_key"])
    labels, particles = segment(overview, fov_m=probe["overview"]["fov_m"])
    if not particles:
        raise SystemExit("no particle in this overview: move to an area with particles and re-run")
    smooth_background = np.unravel_index(np.argmin(overview), overview.shape)
    points = [{"id": 1, "row": particles[0]["row"], "col": particles[0]["col"]},
              {"id": 2, "row": float(smooth_background[0]), "col": float(smooth_background[1])}]
    plot_particles(overview, labels, points[:1], "1 = particle (background = darkest pixel)")
    results = {}
    for order in ("col_row", "row_col"):
        t0 = time.time()
        update = await eds_node({**probe, "particles": points,
                                 "settings": {**SETTINGS, "beam_order": order}})
        particle, background = (signal(r) for r in update["spectra"])
        results[order] = particle / max(background, 1e-12)
        print(f"{order}: particle {particle:.3g}, background {background:.3g}, "
              f"ratio {results[order]:.2f} ({(time.time() - t0) / 2:.1f} s per spectrum)")
    best = max(results, key=results.get)
    print(f"-> beam order looks like {best!r}; configured {TARGETS[TARGET]['beam_order']!r}")

## Level 4 · Tuning nodes
Each runs `QuadraticSearch` on the microscope with the small `TEST_TUNING` settings, ends at the best point, and saves `tune_<name>.json` and `.png` in the run folder. On error it restores its start value.

### 4.1 Focus (stage z)

In [ ]:
if allowed(ALLOW_STAGE, "focus z"):
    _, start_image = await common.acquire_haadf()
    state = apply(state, await focus_z_node(state))
    if "z" in state.get("tuning", {}):
        result = state["tuning"]["z"]
        show([start_image, tiled_io.read_image(result["best_image_key"])], ["start", "best z"])
        display(Image(result["figure_path"]))

### 4.2 C1 (at the tuning FOV)

In [ ]:
if allowed(ALLOW_CORRECTOR, "tune C1"):
    state = apply(state, await set_fov_node(state))
    state = apply(state, await tune_c1_node(state))
    if "C1" in state.get("tuning", {}):
        display(Image(state["tuning"]["C1"]["figure_path"]))

### 4.3 A1

In [ ]:
if allowed(ALLOW_CORRECTOR, "tune A1"):
    state = apply(state, await tune_a1_node(state))
    if "A1" in state.get("tuning", {}):
        display(Image(state["tuning"]["A1"]["figure_path"]))

### 4.4 Put the tuning back (optional)
Returns z, C1 and A1 to where they were at status start. Use it if you want the microscope exactly as before.

In [ ]:
if allowed(ALLOW_STAGE and ALLOW_CORRECTOR, "undo all tuning"):
    await common.move_stage(z=state["start"]["stage"][2])
    for name, key in (("C1", "c1_offset_m"), ("A1", "a1_offset_m")):
        total = np.atleast_1d(state["current"].get(key, 0.0))
        if np.any(total):
            corrector = Corrector(name)
            corrector.total = total
            await corrector.restore()
    print("z, C1 and A1 back at the status-start values")

## Level 5 · Segmentation and EDS

### 5.1 Overview and segmentation

In [ ]:
state = apply(state, await set_fov_node(state))
state = apply(state, await overview_node(state))
state = apply(state, await segment_node(state))
Image(state["segmentation"]["overlay_path"])

### 5.2 Try other segmentation settings (no new image)
Settings are in nm. Pick the one that outlines the particles best, then keep it in `SETTINGS["segmentation"]`.

In [ ]:
overview = tiled_io.read_image(state["overview"]["tiled_key"])
fov_now = state["overview"]["fov_m"]
for options in ({"smooth_nm": 0.3, "min_diameter_nm": 1.0},
                {"smooth_nm": 0.7, "min_diameter_nm": 2.9},     # the original notebook's values at 186 nm
                {"smooth_nm": 1.5, "min_diameter_nm": 5.0}):
    labels, particles = segment(overview, fov_now, **options)
    display(plot_particles(overview, labels, particles, f"{options}: {len(particles)} particles"))
# SETTINGS["segmentation"] = {"smooth_nm": 0.7, "min_diameter_nm": 2.9}

### 5.3 EDS on the first particles
At most `MAX_EDS_POINTS` spectra.

In [ ]:
if allowed(ALLOW_BEAM, "EDS on particles"):
    state = apply(state, await eds_node(state))
    display(Image(Path(state["run"]["run_dir"]) / "eds.png"))

### Close the hand-made run
Status end and save: the same record a graph run leaves.

In [ ]:
state = apply(state, await status_end_node(state))
state = apply(state, await save_node(state))
Image(Path(state["run"]["run_dir"]) / "summary.png")

## Level 6 · Build your own graph
Every node is a plain async function `state -> update`, so any order works. Use this to try a partial workflow on the microscope.

The example: **overview → segment → EDS**, with no tuning. It has the same safety pattern as the full graph: a router after each node jumps to `status_end` on failure or when no particles are found, and `status_end → save` always run.

In [ ]:
from langgraph.graph import END, START, StateGraph

from stem_agent.states.experiment_state import ParticleRunState


def next_or_stop(next_node):
    """Router: go on, or jump to status_end when the run failed or found no particles."""
    def route(state):
        return "status_end" if state.get("run_status") in ("failed", "no_particles") else next_node
    return route


steps = [("preflight", preflight_node),
         ("status_start", status_start_node),
         ("set_fov", set_fov_node),
         ("overview", overview_node),
         ("segment", segment_node),
         ("eds", eds_node)]                     # add / remove / reorder nodes here

builder = StateGraph(ParticleRunState)
for name, node in steps:
    builder.add_node(name, node)
builder.add_node("status_end", status_end_node)
builder.add_node("save", save_node)

builder.add_edge(START, steps[0][0])
for (name, _), (next_name, _) in zip(steps, steps[1:]):
    builder.add_conditional_edges(name, next_or_stop(next_name), [next_name, "status_end"])
builder.add_edge(steps[-1][0], "status_end")
builder.add_edge("status_end", "save")
builder.add_edge("save", END)

my_graph = builder.compile()
plot_graph(my_graph)

In [ ]:
needs_beam = any(name == "eds" for name, _ in steps)
if allowed(ALLOW_BEAM or not needs_beam, "custom graph (it takes EDS spectra)"):
    my_state = await my_graph.ainvoke({"settings": {**SETTINGS, "run_tag": "custom_graph"}})
    print("run_status:", my_state["run_status"], "| errors:", my_state.get("errors") or "none")
    display(Image(Path(my_state["run"]["run_dir"]) / "summary.png"))

## Level 7 · Full run
The whole graph (`particle_graph.run`) with the small `TEST_TUNING` settings, at the current stage position. It needs all three switches.

In [ ]:
from stem_agent.graphs.particle_graph import build_graph, run

plot_graph(build_graph())

In [ ]:
if allowed(ALLOW_STAGE and ALLOW_CORRECTOR and ALLOW_BEAM, "full run"):
    full = await run({**SETTINGS, "run_tag": "real_full"})
    print("run_status:", full["run_status"], "| errors:", full.get("errors") or "none")
    display(Image(Path(full["run"]["run_dir"]) / "summary.png"))

## Notes for the session log
Write down:
- the tool names, if 1.2 needed fixes
- the alpha / `NaN` result (1.3, 2.4)
- whether the metrics respond (3.1, 3.2) and at which step sizes
- the corrector sign (3.3)
- the beam order (3.4)
- the HAADF and EDS times (1.5, 3.4)

The beam order and any tool-name fixes go into `stem_agent/config.py` (`TARGETS["real"]`).